# SAM2 Parameter Optimization for Cell Counting
Find optimal SAM2 parameters using MAPE between predicted and ground truth cell counts.

Run this notebook from within the SAM2 directory where their other notebooks are located.

In [1]:
import numpy as np
import torch
import cv2
import os
from pathlib import Path
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.build_sam import build_sam2
import json
import pandas as pd
from itertools import product
import time

In [2]:
# specify a gpu to use
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
#    Constants
# -------------------------------------------------------------------- #

# acceptable image file extensions
IMG_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.TIF']

# SAM2 constants
SAM2_CHECKPOINT = "/home/username/Models/sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
DEVICE = "cuda"

# Sampled dataset paths
W1_IMAGES_FOLDER = '/home/username/Data/Microscopy/BBBC005/sampled_images/w1/images'
W1_GT_FOLDER = '/home/username/Data/Microscopy/BBBC005/sampled_images/w1/ground_truth'
W2_IMAGES_FOLDER = '/home/username/Data/Microscopy/BBBC005/sampled_images/w2/images'
W2_GT_FOLDER = '/home/username/Data/Microscopy/BBBC005/sampled_images/w2/ground_truth'

# Results
STUDY_FOLDER = '/home/username/Code/SAM2/pv_results/parameter_optimization/'
os.makedirs(STUDY_FOLDER, exist_ok=True)

## Define methods

In [4]:
def show_anns_binary(anns):
    """Extract filtered mask count from SAM2 annotations"""
    if len(anns) == 0:
        return 0
    
    # Sort by area and filter large masks (>25% of image area)
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    if len(sorted_anns) > 0:
        img_area = sorted_anns[0]['segmentation'].shape[0] * sorted_anns[0]['segmentation'].shape[1]
        sorted_anns = [ann for ann in sorted_anns if ann['area'] < 0.25 * img_area]
    
    return len(sorted_anns)

def extract_ground_truth_count(filename):
    """Extract ground truth cell count from filename like 'SIMCEPImages_A01_C1_F1_s01_w1.png'"""
    parts = filename.split("_")
    for part in parts:
        if part.startswith("C") and len(part) > 1:
            return int(part[1:])  # Extract number after 'C'
    return 0

def calculate_mape(predicted, ground_truth):
    """Calculate Mean Absolute Percentage Error"""
    # Avoid division by zero
    mask = ground_truth > 0
    if not np.any(mask):
        return float('inf')
    
    return np.mean(np.abs((predicted[mask] - ground_truth[mask]) / ground_truth[mask])) * 100

def load_dataset(images_folder, gt_folder, image_type):
    """Load images and extract ground truth counts from filenames"""
    images = []
    gt_counts = []
    filenames = []
    
    for file in sorted(os.listdir(images_folder)):
        if any([file.endswith(ext) for ext in IMG_EXTENSIONS]):
            # Load image
            img_path = os.path.join(images_folder, file)
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            images.append(image)
            
            # Extract ground truth count from filename
            gt_count = extract_ground_truth_count(file)
            gt_counts.append(gt_count)
            filenames.append(file)
    
    print(f"Loaded {len(images)} {image_type} images")
    return images, np.array(gt_counts), filenames

In [5]:
def evaluate_parameters(sam2_model, images, gt_counts, params, image_type):
    """Evaluate SAM2 parameters on dataset and return MAPE"""
    
    # Create mask generator with specific parameters
    mask_generator = SAM2AutomaticMaskGenerator(
        model=sam2_model,
        points_per_side=params['points_per_side'],
        pred_iou_thresh=params['pred_iou_thresh'],
        stability_score_thresh=params['stability_score_thresh'],
        box_nms_thresh=params.get('box_nms_thresh', 0.7),
        min_mask_region_area=params.get('min_mask_region_area', 0)
    )
    
    predicted_counts = []
    
    for i, image in enumerate(images):
        # Generate masks
        masks = mask_generator.generate(image)
        
        # Get filtered count
        predicted_count = show_anns_binary(masks)
        predicted_counts.append(predicted_count)
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(images)} {image_type} images")
    
    predicted_counts = np.array(predicted_counts)
    
    # Calculate MAPE
    mape = calculate_mape(predicted_counts, gt_counts)
    
    return mape, predicted_counts

## Load datasets

In [6]:
# Load W1 and W2 datasets
w1_images, w1_gt_counts, w1_filenames = load_dataset(W1_IMAGES_FOLDER, W1_GT_FOLDER, "W1")
w2_images, w2_gt_counts, w2_filenames = load_dataset(W2_IMAGES_FOLDER, W2_GT_FOLDER, "W2")

print(f"W1 ground truth range: {w1_gt_counts.min()} - {w1_gt_counts.max()}")
print(f"W2 ground truth range: {w2_gt_counts.min()} - {w2_gt_counts.max()}")

Loaded 50 W1 images
Loaded 50 W2 images
W1 ground truth range: 1 - 100
W2 ground truth range: 1 - 100


## Initialize SAM2 model

In [7]:
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)

## Parameter optimization

In [8]:
# Define parameter grid for optimization
param_grid = {
    'points_per_side': [16, 32, 64],
    'pred_iou_thresh': [0.7, 0.8, 0.9],
    'stability_score_thresh': [0.90, 0.95, 0.98]
}

# Generate all parameter combinations
param_combinations = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

print(f"Testing {len(param_combinations)} parameter combinations")
print(f"Parameters: {param_names}")

Testing 27 parameter combinations
Parameters: ['points_per_side', 'pred_iou_thresh', 'stability_score_thresh']


In [ ]:
# Initialize results storage
results = []

# Test each parameter combination
for i, param_values in enumerate(param_combinations):
    params = dict(zip(param_names, param_values))
    
    print(f"\n--- Combination {i+1}/{len(param_combinations)} ---")
    print(f"Parameters: {params}")
    
    start_time = time.time()
    
    # Evaluate on W1 images
    print("Evaluating W1 images...")
    w1_mape, w1_pred = evaluate_parameters(sam2_model, w1_images, w1_gt_counts, params, "W1")
    
    # Evaluate on W2 images  
    print("Evaluating W2 images...")
    w2_mape, w2_pred = evaluate_parameters(sam2_model, w2_images, w2_gt_counts, params, "W2")
    
    elapsed_time = time.time() - start_time
    
    # Store results
    result = {
        'combination': i + 1,
        'points_per_side': params['points_per_side'],
        'pred_iou_thresh': params['pred_iou_thresh'],
        'stability_score_thresh': params['stability_score_thresh'],
        'w1_mape': w1_mape,
        'w2_mape': w2_mape,
        'combined_mape': (w1_mape + w2_mape) / 2,
        'elapsed_time': elapsed_time
    }
    results.append(result)
    
    print(f"W1 MAPE: {w1_mape:.3f}%")
    print(f"W2 MAPE: {w2_mape:.3f}%") 
    print(f"Combined MAPE: {result['combined_mape']:.3f}%")
    print(f"Time: {elapsed_time:.1f}s")
    
    # Save intermediate results
    df_results = pd.DataFrame(results)
    df_results.to_csv(os.path.join(STUDY_FOLDER, 'parameter_optimization_results.csv'), index=False)

print(f"\nOptimization complete! Results saved to {STUDY_FOLDER}")

The following result was obtained after running the above cell.

```
--- Combination 1/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 17.475%
W2 MAPE: 28.339%
Combined MAPE: 22.907%
Time: 70.4s

--- Combination 2/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 26.142%
W2 MAPE: 43.736%
Combined MAPE: 34.939%
Time: 62.6s

--- Combination 3/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 99.115%
W2 MAPE: 94.513%
Combined MAPE: 96.814%
Time: 55.4s

--- Combination 4/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 18.765%
W2 MAPE: 31.301%
Combined MAPE: 25.033%
Time: 69.8s

--- Combination 5/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 26.434%
W2 MAPE: 44.830%
Combined MAPE: 35.632%
Time: 62.5s

--- Combination 6/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 99.115%
W2 MAPE: 94.513%
Combined MAPE: 96.814%
Time: 55.3s

--- Combination 7/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 26.153%
W2 MAPE: 39.756%
Combined MAPE: 32.954%
Time: 69.0s

--- Combination 8/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 29.545%
W2 MAPE: 47.547%
Combined MAPE: 38.546%
Time: 62.3s

--- Combination 9/27 ---
Parameters: {'points_per_side': 16, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 99.115%
W2 MAPE: 94.513%
Combined MAPE: 96.814%
Time: 55.2s

--- Combination 10/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.844%
W2 MAPE: 20.353%
Combined MAPE: 12.598%
Time: 213.6s

--- Combination 11/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.834%
W2 MAPE: 39.649%
Combined MAPE: 22.242%
Time: 185.1s

--- Combination 12/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 97.944%
W2 MAPE: 92.394%
Combined MAPE: 95.169%
Time: 157.7s

--- Combination 13/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.577%
W2 MAPE: 21.967%
Combined MAPE: 13.272%
Time: 212.9s

--- Combination 14/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.876%
W2 MAPE: 40.491%
Combined MAPE: 22.684%
Time: 184.6s

--- Combination 15/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 97.944%
W2 MAPE: 92.394%
Combined MAPE: 95.169%
Time: 157.5s

--- Combination 16/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 5.480%
W2 MAPE: 33.979%
Combined MAPE: 19.730%
Time: 210.6s

--- Combination 17/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 6.049%
W2 MAPE: 43.232%
Combined MAPE: 24.641%
Time: 183.7s

--- Combination 18/27 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 97.944%
W2 MAPE: 92.394%
Combined MAPE: 95.169%
Time: 157.0s

--- Combination 19/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 9.434%
W2 MAPE: 19.350%
Combined MAPE: 14.392%
Time: 789.9s

--- Combination 20/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.061%
W2 MAPE: 32.780%
Combined MAPE: 18.420%
Time: 673.9s

--- Combination 21/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 95.595%
W2 MAPE: 91.314%
Combined MAPE: 93.454%
Time: 565.7s

--- Combination 22/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 7.759%
W2 MAPE: 18.586%
Combined MAPE: 13.173%
Time: 787.9s

--- Combination 23/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.133%
W2 MAPE: 33.479%
Combined MAPE: 18.806%
Time: 672.3s

--- Combination 24/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 95.595%
W2 MAPE: 91.314%
Combined MAPE: 93.454%
Time: 565.0s

--- Combination 25/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.9}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 5.592%
W2 MAPE: 26.652%
Combined MAPE: 16.122%
Time: 777.5s

--- Combination 26/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.95}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 5.073%
W2 MAPE: 37.153%
Combined MAPE: 21.113%
Time: 668.5s

--- Combination 27/27 ---
Parameters: {'points_per_side': 64, 'pred_iou_thresh': 0.9, 'stability_score_thresh': 0.98}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 95.679%
W2 MAPE: 91.314%
Combined MAPE: 93.496%
Time: 562.6s
```

## Results analysis

In [10]:
# Load and analyze results
df_results = pd.DataFrame(results)

# Find best parameters for each image type
best_w1_idx = df_results['w1_mape'].idxmin()
best_w2_idx = df_results['w2_mape'].idxmin()
best_combined_idx = df_results['combined_mape'].idxmin()

print("=== OPTIMIZATION RESULTS ===\n")

print("BEST PARAMETERS FOR W1 IMAGES:")
best_w1 = df_results.iloc[best_w1_idx]
print(f"  points_per_side: {int(best_w1['points_per_side'])}")
print(f"  pred_iou_thresh: {best_w1['pred_iou_thresh']}")
print(f"  stability_score_thresh: {best_w1['stability_score_thresh']}")
print(f"  MAPE: {best_w1['w1_mape']:.3f}%")

print("\nBEST PARAMETERS FOR W2 IMAGES:")
best_w2 = df_results.iloc[best_w2_idx]
print(f"  points_per_side: {int(best_w2['points_per_side'])}")
print(f"  pred_iou_thresh: {best_w2['pred_iou_thresh']}")
print(f"  stability_score_thresh: {best_w2['stability_score_thresh']}")
print(f"  MAPE: {best_w2['w2_mape']:.3f}%")

print("\nBEST PARAMETERS FOR COMBINED:")
best_combined = df_results.iloc[best_combined_idx]
print(f"  points_per_side: {int(best_combined['points_per_side'])}")
print(f"  pred_iou_thresh: {best_combined['pred_iou_thresh']}")
print(f"  stability_score_thresh: {best_combined['stability_score_thresh']}")
print(f"  Combined MAPE: {best_combined['combined_mape']:.3f}%")
print(f"  W1 MAPE: {best_combined['w1_mape']:.3f}%")
print(f"  W2 MAPE: {best_combined['w2_mape']:.3f}%")

=== OPTIMIZATION RESULTS ===

BEST PARAMETERS FOR W1 IMAGES:
  points_per_side: 64
  pred_iou_thresh: 0.7
  stability_score_thresh: 0.95
  MAPE: 4.061%

BEST PARAMETERS FOR W2 IMAGES:
  points_per_side: 64
  pred_iou_thresh: 0.8
  stability_score_thresh: 0.9
  MAPE: 18.586%

BEST PARAMETERS FOR COMBINED:
  points_per_side: 32
  pred_iou_thresh: 0.7
  stability_score_thresh: 0.9
  Combined MAPE: 12.598%
  W1 MAPE: 4.844%
  W2 MAPE: 20.353%


In [11]:
# Show top 3 results for each metric
print("\n=== TOP 3 RESULTS ===\n")

print("TOP 3 FOR W1 IMAGES:")
top_w1 = df_results.nsmallest(3, 'w1_mape')[['points_per_side', 'pred_iou_thresh', 'stability_score_thresh', 'w1_mape']]
for idx, row in top_w1.iterrows():
    print(f"  {int(row['points_per_side'])}, {row['pred_iou_thresh']}, {row['stability_score_thresh']} → {row['w1_mape']:.3f}%")

print("\nTOP 3 FOR W2 IMAGES:")
top_w2 = df_results.nsmallest(3, 'w2_mape')[['points_per_side', 'pred_iou_thresh', 'stability_score_thresh', 'w2_mape']]
for idx, row in top_w2.iterrows():
    print(f"  {int(row['points_per_side'])}, {row['pred_iou_thresh']}, {row['stability_score_thresh']} → {row['w2_mape']:.3f}%")

print("\nTOP 3 FOR COMBINED:")
top_combined = df_results.nsmallest(3, 'combined_mape')[['points_per_side', 'pred_iou_thresh', 'stability_score_thresh', 'combined_mape']]
for idx, row in top_combined.iterrows():
    print(f"  {int(row['points_per_side'])}, {row['pred_iou_thresh']}, {row['stability_score_thresh']} → {row['combined_mape']:.3f}%")


=== TOP 3 RESULTS ===

TOP 3 FOR W1 IMAGES:
  64, 0.7, 0.95 → 4.061%
  64, 0.8, 0.95 → 4.133%
  32, 0.8, 0.9 → 4.577%

TOP 3 FOR W2 IMAGES:
  64, 0.8, 0.9 → 18.586%
  64, 0.7, 0.9 → 19.350%
  32, 0.7, 0.9 → 20.353%

TOP 3 FOR COMBINED:
  32, 0.7, 0.9 → 12.598%
  64, 0.8, 0.9 → 13.173%
  32, 0.8, 0.9 → 13.272%


## Final Parameter Search - SAM1/SAM2 Union with Standard Baseline

In [12]:
# Final parameter grid with standard baseline
final_param_grid = {
    'points_per_side': [32],
    'pred_iou_thresh': [0.7, 0.8, 0.9, 0.98],
    'stability_score_thresh': [0.9, 0.95, 0.98],
    'box_nms_thresh': [0.7, 0.8, 0.9],
    'crop_nms_thresh': [0.7, 0.8, 0.95],
    'crop_overlap_ratio': [0.05, 0.3]
}

# Add standard SAM2 parameters as baseline
standard_params = {
    'points_per_side': 32,
    'pred_iou_thresh': 0.8,      # SAM2 default
    'stability_score_thresh': 0.95,  # SAM2 default
    'box_nms_thresh': 0.7,      # SAM2 default
    'crop_nms_thresh': 0.7,     # SAM2 default
    'crop_overlap_ratio': 512/1500  # SAM2 default ≈ 0.34
}

# Generate all combinations from grid
final_combinations = []
for pps in final_param_grid['points_per_side']:
    for pit in final_param_grid['pred_iou_thresh']:
        for sst in final_param_grid['stability_score_thresh']:
            for bnt in final_param_grid['box_nms_thresh']:
                for cnt in final_param_grid['crop_nms_thresh']:
                    for cor in final_param_grid['crop_overlap_ratio']:
                        final_combinations.append((pps, pit, sst, bnt, cnt, cor))

# Add standard parameters as first combination
final_combinations.insert(0, (
    standard_params['points_per_side'],
    standard_params['pred_iou_thresh'],
    standard_params['stability_score_thresh'],
    standard_params['box_nms_thresh'],
    standard_params['crop_nms_thresh'],
    standard_params['crop_overlap_ratio']
))

final_param_names = ['points_per_side', 'pred_iou_thresh', 'stability_score_thresh', 
                     'box_nms_thresh', 'crop_nms_thresh', 'crop_overlap_ratio']

print(f"Total combinations to test: {len(final_combinations)}")
print(f"Parameters: {final_param_names}")
print(f"First combination is SAM2 standard parameters (baseline)")
print(f"Remaining {len(final_combinations)-1} combinations from union grid")

Total combinations to test: 217
Parameters: ['points_per_side', 'pred_iou_thresh', 'stability_score_thresh', 'box_nms_thresh', 'crop_nms_thresh', 'crop_overlap_ratio']
First combination is SAM2 standard parameters (baseline)
Remaining 216 combinations from union grid


In [13]:
# Final evaluation function with all parameters
def evaluate_parameters_final(sam2_model, images, gt_counts, params, image_type):
    """Evaluate SAM2 parameters including all crop parameters on dataset and return MAPE"""
    
    # Create mask generator with all parameters
    mask_generator = SAM2AutomaticMaskGenerator(
        model=sam2_model,
        points_per_side=params['points_per_side'],
        pred_iou_thresh=params['pred_iou_thresh'],
        stability_score_thresh=params['stability_score_thresh'],
        box_nms_thresh=params['box_nms_thresh'],
        crop_nms_thresh=params['crop_nms_thresh'],
        crop_overlap_ratio=params['crop_overlap_ratio'],
        min_mask_region_area=0
    )
    
    predicted_counts = []
    
    for i, image in enumerate(images):
        # Generate masks
        masks = mask_generator.generate(image)
        
        # Get filtered count
        predicted_count = show_anns_binary(masks)
        predicted_counts.append(predicted_count)
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(images)} {image_type} images")
    
    predicted_counts = np.array(predicted_counts)
    
    # Calculate MAPE
    mape = calculate_mape(predicted_counts, gt_counts)
    
    return mape, predicted_counts

In [14]:
# Run final parameter optimization
final_results = []

print("=== STARTING FINAL OPTIMIZATION ===")
print(f"Testing {len(final_combinations)} combinations")
print("Combination 1: SAM2 Standard (baseline)")
print(f"Combinations 2-{len(final_combinations)}: SAM1/SAM2 Union Grid\n")

# Test each final parameter combination
for i, param_values in enumerate(final_combinations):
    params = dict(zip(final_param_names, param_values))
    
    combo_type = "STANDARD" if i == 0 else "UNION"
    print(f"\n--- {combo_type} Combination {i+1}/{len(final_combinations)} ---")
    print(f"Parameters: {params}")
    
    start_time = time.time()
    
    # Evaluate on W1 images
    print("Evaluating W1 images...")
    w1_mape, w1_pred = evaluate_parameters_final(sam2_model, w1_images, w1_gt_counts, params, "W1")
    
    # Evaluate on W2 images  
    print("Evaluating W2 images...")
    w2_mape, w2_pred = evaluate_parameters_final(sam2_model, w2_images, w2_gt_counts, params, "W2")
    
    elapsed_time = time.time() - start_time
    
    # Store results (no combined MAPE)
    result = {
        'combination': i + 1,
        'type': combo_type,
        'points_per_side': params['points_per_side'],
        'pred_iou_thresh': params['pred_iou_thresh'],
        'stability_score_thresh': params['stability_score_thresh'],
        'box_nms_thresh': params['box_nms_thresh'],
        'crop_nms_thresh': params['crop_nms_thresh'],
        'crop_overlap_ratio': params['crop_overlap_ratio'],
        'w1_mape': w1_mape,
        'w2_mape': w2_mape,
        'elapsed_time': elapsed_time
    }
    final_results.append(result)
    
    print(f"W1 MAPE: {w1_mape:.3f}%")
    print(f"W2 MAPE: {w2_mape:.3f}%") 
    print(f"Time: {elapsed_time:.1f}s")
    
    # Save intermediate results to new CSV
    df_final = pd.DataFrame(final_results)
    df_final.to_csv(os.path.join(STUDY_FOLDER, 'final_parameter_optimization_results.csv'), index=False)

print(f"\nFinal optimization complete! Results saved to {STUDY_FOLDER}/final_parameter_optimization_results.csv")

=== STARTING FINAL OPTIMIZATION ===
Testing 217 combinations
Combination 1: SAM2 Standard (baseline)
Combinations 2-217: SAM1/SAM2 Union Grid


--- STANDARD Combination 1/217 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.8, 'stability_score_thresh': 0.95, 'box_nms_thresh': 0.7, 'crop_nms_thresh': 0.7, 'crop_overlap_ratio': 0.3413333333333333}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/50 W1 images
  Processed 30/50 W1 images
  Processed 40/50 W1 images
  Processed 50/50 W1 images
Evaluating W2 images...
  Processed 10/50 W2 images
  Processed 20/50 W2 images
  Processed 30/50 W2 images
  Processed 40/50 W2 images
  Processed 50/50 W2 images
W1 MAPE: 4.876%
W2 MAPE: 40.491%
Time: 185.2s

--- UNION Combination 2/217 ---
Parameters: {'points_per_side': 32, 'pred_iou_thresh': 0.7, 'stability_score_thresh': 0.9, 'box_nms_thresh': 0.7, 'crop_nms_thresh': 0.7, 'crop_overlap_ratio': 0.05}
Evaluating W1 images...
  Processed 10/50 W1 images
  Processed 20/

## Simple Results

In [15]:
# Simple final results analysis
df_final = pd.DataFrame(final_results)

# Find best parameters for W1 and W2 separately
best_w1_idx = df_final['w1_mape'].idxmin()
best_w2_idx = df_final['w2_mape'].idxmin()

print("=== FINAL RESULTS ===\n")

print("STANDARD SAM2 PARAMETERS (BASELINE):")
standard_result = df_final[df_final['type'] == 'STANDARD'].iloc[0]
print(f"  W1 MAPE: {standard_result['w1_mape']:.3f}%")
print(f"  W2 MAPE: {standard_result['w2_mape']:.3f}%")

print("\nBEST PARAMETERS FOR W1 IMAGES:")
best_w1 = df_final.iloc[best_w1_idx]
print(f"  pred_iou_thresh: {best_w1['pred_iou_thresh']}")
print(f"  stability_score_thresh: {best_w1['stability_score_thresh']}")
print(f"  box_nms_thresh: {best_w1['box_nms_thresh']}")
print(f"  crop_nms_thresh: {best_w1['crop_nms_thresh']}")
print(f"  crop_overlap_ratio: {best_w1['crop_overlap_ratio']}")
print(f"  W1 MAPE: {best_w1['w1_mape']:.3f}%")

print("\nBEST PARAMETERS FOR W2 IMAGES:")
best_w2 = df_final.iloc[best_w2_idx]
print(f"  pred_iou_thresh: {best_w2['pred_iou_thresh']}")
print(f"  stability_score_thresh: {best_w2['stability_score_thresh']}")
print(f"  box_nms_thresh: {best_w2['box_nms_thresh']}")
print(f"  crop_nms_thresh: {best_w2['crop_nms_thresh']}")
print(f"  crop_overlap_ratio: {best_w2['crop_overlap_ratio']}")
print(f"  W2 MAPE: {best_w2['w2_mape']:.3f}%")

print(f"\nResults saved to: {STUDY_FOLDER}/final_parameter_optimization_results.csv")

=== FINAL RESULTS ===

STANDARD SAM2 PARAMETERS (BASELINE):
  W1 MAPE: 4.876%
  W2 MAPE: 40.491%

BEST PARAMETERS FOR W1 IMAGES:
  pred_iou_thresh: 0.8
  stability_score_thresh: 0.9
  box_nms_thresh: 0.7
  crop_nms_thresh: 0.7
  crop_overlap_ratio: 0.05
  W1 MAPE: 4.577%

BEST PARAMETERS FOR W2 IMAGES:
  pred_iou_thresh: 0.7
  stability_score_thresh: 0.9
  box_nms_thresh: 0.9
  crop_nms_thresh: 0.7
  crop_overlap_ratio: 0.05
  W2 MAPE: 17.585%

Results saved to: /home/prateek/Code/SAM2/pv_results/parameter_optimization//final_parameter_optimization_results.csv
